# V2a-RSN 220119_F2_run11 — Attribution and Feature Sensitivity Notebook: c-GC*

Prerequisites: `community`, `manifold`, `cgc_star` `connectivity`, and `linking` stages, plus the dataset `shared/recording` checkpoint.
This notebook reports predictor-feature sensitivity only; it does not identify a biological intervention.
Progress: long stages print checkpoint-aware timing; iterative stages also display live progress bars. Reused, failed, running, and missing checkpoints are reported explicitly.


In [ ]:
import sys
from pathlib import Path

try:
    _REPOSITORY_ROOT = next(
        path
        for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "notebooks" / "_shared.py").is_file()
    )
except StopIteration as error:
    raise RuntimeError(
        "Could not locate the effectome repository from the notebook working directory"
    ) from error
sys.path[:0] = [
    path
    for path in (str(_REPOSITORY_ROOT / "src"), str(_REPOSITORY_ROOT))
    if path not in sys.path
]


In [ ]:
from notebooks._shared import (
    PROJECT_ROOT,
    make_run,
    print_notebook_header,
    print_stage_status,
    run_candidate_drivers,
    run_feature_sensitivity,
)

DATASET_ID = "220119_F2_run11"
DATASET_OUTPUT_ID = "v2a-rsns"
RECORDING_ID = "220119_F2_run11"
RUN_ID = "reference"
METHOD = "cgc_star"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "analysis" / DATASET_OUTPUT_ID / RECORDING_ID
RECORDING_PATH = OUTPUT_ROOT / RUN_ID / "shared" / "stages" / "recording" / "artifact.pkl"
BEHAVIOR_KEY = "continuous"
FORCE = False

run = make_run(OUTPUT_ROOT, RUN_ID, METHOD)
print_notebook_header(
    run,
    title=f"{METHOD} attribution and feature sensitivity",
    expected_stages=[
        "recording_input",
        "candidate_drivers",
        "surrogate_model",
        "feature_sensitivity",
    ],
    force=FORCE,
)


In [ ]:
candidates = run_candidate_drivers(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=1,
    n_folds=5,
    embargo=4,
    seed=42,
    matched_control_percentile=95.0,
    force=FORCE,
)
top = candidates.scores[0]
print(top)
print_stage_status(run)


In [ ]:
sensitivity = run_feature_sensitivity(
    run,
    recording_path=RECORDING_PATH,
    behavior_key=BEHAVIOR_KEY,
    lag=1,
    ridge_alpha=1.0,
    n_folds=5,
    embargo=4,
    min_skill=0.0,
    scale=0.0,
    dose_scales=[1.0, 0.5, 0.0],
    top_k=1,
    endpoint="behavior",
    seed=42,
    force=FORCE,
)
print(sensitivity)
print(sensitivity.analysis_kind)
print(sensitivity.biological_intervention_identified)
print_stage_status(run)
